# Analyse exploratoire · ChestMNIST

**Projet Deep Learning · EFREI M1 Data Engineering & IA**
**Trinôme · Adam Beloucif · Emilien Morice · Arnaud Dissongo**

Section 3 du RAPPORT · distribution des 14 pathologies, co-occurrence,
exemples positifs, statistiques de support. Le notebook s'exécute en
moins d'une minute sur CPU.

Pré-requis · `python -m scripts.download_chestmnist --sizes 64` doit avoir
été lancé au moins une fois pour que `data/chestmnist_64.npz` existe.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 110

from src.config import CHEST_LABELS, NUM_CLASSES
print(f'{NUM_CLASSES} classes ·', CHEST_LABELS)

## 1. Chargement

In [ ]:
import medmnist

try:
    ds_train = medmnist.ChestMNIST(split='train', download=False, size=64,
                                   root=str(ROOT / 'data'))
    ds_val   = medmnist.ChestMNIST(split='val',   download=False, size=64,
                                   root=str(ROOT / 'data'))
    ds_test  = medmnist.ChestMNIST(split='test',  download=False, size=64,
                                   root=str(ROOT / 'data'))
except TypeError:
    ds_train = medmnist.ChestMNIST(split='train', download=False, root=str(ROOT / 'data'))
    ds_val   = medmnist.ChestMNIST(split='val',   download=False, root=str(ROOT / 'data'))
    ds_test  = medmnist.ChestMNIST(split='test',  download=False, root=str(ROOT / 'data'))

X_train, y_train = np.asarray(ds_train.imgs), np.asarray(ds_train.labels)
X_val,   y_val   = np.asarray(ds_val.imgs),   np.asarray(ds_val.labels)
X_test,  y_test  = np.asarray(ds_test.imgs),  np.asarray(ds_test.labels)

print('train', X_train.shape, y_train.shape)
print('val  ', X_val.shape,   y_val.shape)
print('test ', X_test.shape,  y_test.shape)

## 2. Distribution des labels (train)

In [ ]:
support = y_train.sum(axis=0).astype(int)
df_lbl = pd.DataFrame({'pathology': CHEST_LABELS, 'support': support})
df_lbl['prevalence'] = df_lbl['support'] / len(y_train)
df_lbl = df_lbl.sort_values('support', ascending=False).reset_index(drop=True)
df_lbl

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=df_lbl.sort_values('support'), y='pathology', x='support',
            color='#001329', ax=ax)
for i, row in enumerate(df_lbl.sort_values('support').itertuples()):
    ax.text(row.support + 200, i, f"{row.support}  ({row.prevalence:.1%})",
            va='center', fontsize=8)
ax.set_title('Distribution des 14 pathologies · ChestMNIST train')
ax.set_xlabel('Nombre d'images positives')
ax.set_ylabel('')
plt.tight_layout()

## 3. Co-occurrence des labels

`P(col | ligne)` · sachant qu'une image porte la pathologie sur la ligne,
quelle est la probabilité qu'elle porte aussi celle de la colonne ?

In [ ]:
cooc = y_train.T @ y_train
diag = np.diag(cooc).astype(float)
norm = cooc / np.where(diag > 0, diag, 1)[:, None]
df_cooc = pd.DataFrame(norm, index=CHEST_LABELS, columns=CHEST_LABELS)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(df_cooc, cmap='Blues', vmin=0, vmax=1, annot=False,
            cbar_kws={'label': 'P(col | ligne)'}, ax=ax)
ax.set_title('Co-occurrence des 14 pathologies')
plt.tight_layout()

Observations cliniques attendues ·
- `effusion` ↔ `atelectasis` souvent ensemble (épanchement comprime le lobe)
- `infiltration` co-occurrente avec `consolidation`
- `pneumothorax` peu co-occurrent avec d'autres pathologies (urgence isolée)

## 4. Exemples visuels

In [ ]:
fig, axes = plt.subplots(2, 7, figsize=(15, 4.5))
for idx, ax in enumerate(axes.flatten()):
    pos = np.where(y_train[:, idx] > 0)[0]
    if len(pos) == 0:
        ax.set_title(f'{CHEST_LABELS[idx]}\n0 image', fontsize=9)
        ax.axis('off')
        continue
    pick = pos[0]
    img = X_train[pick]
    ax.imshow(img if img.ndim == 3 else img, cmap='gray')
    ax.set_title(f'{CHEST_LABELS[idx]}\n{len(pos)} images', fontsize=9)
    ax.axis('off')
plt.tight_layout()

## 5. Vérification splits (anti-fuite)

MedMNIST fournit des splits patient-disjoint construits depuis NIH ChestX-ray14.
On vérifie juste que les tailles ont du sens et que la distribution des
labels reste comparable entre train/val/test.

In [ ]:
splits = {'train': y_train, 'val': y_val, 'test': y_test}
rows = []
for name, y in splits.items():
    rows.append({
        'split': name,
        'n_samples': len(y),
        'pos_rate_macro': float(y.mean()),
        **{lbl: float(y[:, i].mean()) for i, lbl in enumerate(CHEST_LABELS)},
    })
df_split = pd.DataFrame(rows)
df_split.set_index('split').T

## 6. Conclusion EDA

- Distribution **fortement déséquilibrée** · `hernia` (0.18 %) à
  `infiltration` (17.7 %) sur le train. Stratégie de compensation ·
  `pos_weight = N_neg / N_pos` clippé à 20 dans la `BCEWithLogitsLoss`.
- **Co-occurrences cohérentes** avec la littérature (effusion ↔ atelectasis).
  Justifie l'usage de la sigmoïde par classe (multi-label) plutôt qu'une softmax.
- Les splits MedMNIST sont patient-disjoint et stables en proportion d'occurrence ·
  pas de re-split nécessaire.
- À 64×64, certaines pathologies fines (`nodule`) restent visuellement
  difficiles · les expériences à 128 ou 224 px sont attendues pour gagner
  du rappel sur ces classes.